Federal University of Paraiba \
Center of Tecnology \
Program in Civil and Environmental Engineering

**Subject: Export data to barplots of LULC area for each biome in significant rainfall trend areas**


Developed by Jaqueline Coutinho (2026-01-20) \
Last update: 2026-03-16

In [1]:
# Import libraries
import ee
import pandas as pd
import geemap
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.plot import show
from rasterio.mask import mask
import numpy as np

In [2]:
ee.Authenticate()
ee.Initialize(project='ee-jaquelinevigolo')

In [3]:
# Define the region of interest
asset_path = 'users/jaquelinevigolo/lm_bioma_250'
biomes = ee.FeatureCollection(asset_path)  # Load the asset as a FeatureCollection

# # Filter the feature collection by 'id' field for different regions
amazon = biomes.filter(ee.Filter.eq('CD_Bioma', 1))
caatinga = biomes.filter(ee.Filter.eq('CD_Bioma', 2))
cerrado = biomes.filter(ee.Filter.eq('CD_Bioma', 3))
atlanticForest = biomes.filter(ee.Filter.eq('CD_Bioma', 4))
pampa = biomes.filter(ee.Filter.eq('CD_Bioma', 5))
pantanal = biomes.filter(ee.Filter.eq('CD_Bioma', 6))


# asset_path = 'FAO/GAUL/2015/level0'
# brazil = ee.FeatureCollection(asset_path)  # Load the asset as a FeatureCollection

In [4]:
lulc = ee.Image('projects/mapbiomas-public/assets/brazil/lulc/collection10/mapbiomas_brazil_collection10_coverage_v2')
# lulc.getInfo()

In [5]:
start_date=ee.Date('1987-01-01')
end_date=ee.Date('2024-01-01')
years = range(start_date.get('year').getInfo(), end_date.get('year').getInfo())

In [6]:
biomes = {
    # "Amazon": amazon.geometry(),
    # "Caatinga": caatinga.geometry(),
    # "Cerrado": cerrado.geometry(),
    "Atlantic Forest": atlanticForest.geometry(),
    "Pantanal": pantanal.geometry(),
    "Pampa": pampa.geometry()
}

In [7]:
# Your groups
forest = [3, 4, 5, 6, 49]
otherVegetation = [11, 12, 32, 29, 50]
farming = [15, 39, 20, 40, 62, 41, 46, 47, 35, 48, 9, 21]
nonVegetated = [23, 24, 30, 75, 25]
water = [33, 31]
notObserved = [27]

# Group code (you choose any ID)
group_codes = {
    "forest": 1,
    "otherVegetation": 2,
    "farming": 3,
    "nonVegetated": 4,
    "water": 5,
    "notObserved": 6
}

In [8]:
trend_image = ee.Image(
    "projects/ee-jaquelinevigolo/assets/trendPercent_CHIRPS_Rx5day_1987-2023_proj").rename('trendPercent')

In [9]:
def mask_list(image, values):
    # Remap the classes in "values" to 1, everything else to 0
    return image.remap(values, [1] * len(values), 0).eq(1)

def reclass(image):
    base = ee.Image(0)

    base = base.where(mask_list(image, forest), group_codes["forest"])
    base = base.where(mask_list(image, otherVegetation), group_codes["otherVegetation"])
    base = base.where(mask_list(image, farming), group_codes["farming"])
    base = base.where(mask_list(image, nonVegetated), group_codes["nonVegetated"])
    base = base.where(mask_list(image, water), group_codes["water"])
    base = base.where(mask_list(image, notObserved), group_codes["notObserved"])
    
    return base

In [10]:
rows = []
for biome_name, biome_geom in biomes.items():
    for label, code in group_codes.items():
        for trend_sign in ["positive", "negative"]:

            for y in years:
                year = ee.Number(y)
                band = ee.String("classification_").cat(year.format())
                image = lulc.select(band)

                reclassified_lulc = reclass(image)

                # Ensure proper band names
                trend_image = trend_image.rename('trendPercent')

                # Trend mask
                if trend_sign == "positive":
                    trend_mask = trend_image.gt(0)
                else:
                    trend_mask = trend_image.lt(0)

                # Mask pasture years using trend mask
                reclassified_lulc_in_trend = reclassified_lulc.updateMask(trend_mask).rename('lulc_classes').set('year',y)

                pixelArea = ee.Image.pixelArea()

                def calculate_area(image, lulc_code, geometry):
                    mask = image.eq(lulc_code)
                    area = pixelArea.updateMask(mask).reduceRegion(
                        reducer=ee.Reducer.sum(),
                        geometry=geometry,
                        scale=30,
                        bestEffort=True
                    )
                    return ee.Number(area.get("area")).divide(1e6)  # m² → km²

                


                area_km2 = calculate_area(reclassified_lulc_in_trend, code, biome_geom).getInfo()
                
                rows.append({
                    "year": y,
                    "biome": biome_name,
                    "trend_sign": trend_sign,
                    "code": code,
                    "name": label, 
                    "area_km2": area_km2
                })

df = pd.DataFrame(rows)

df.to_csv(r"D:\VSCode\MapBiomas-Award\MapBiomas_Area_ByBiome_ByTrend_Rx5day_1987-2023_proj-1.csv", index=False)

df

,year,biome,trend_sign,code,name,area_km2
0,1987,Atlantic Forest,positive,1,forest,716.371875
1,1988,Atlantic Forest,positive,1,forest,702.961612
2,1989,Atlantic Forest,positive,1,forest,683.871384
3,1990,Atlantic Forest,positive,1,forest,668.790344
4,1991,Atlantic Forest,positive,1,forest,648.497676
...,...,...,...,...,...,...
1327,2019,Pampa,negative,6,notObserved,0.000000
1328,2020,Pampa,negative,6,notObserved,0.000000
1329,2021,Pampa,negative,6,notObserved,0.000000
1330,2022,Pampa,negative,6,notObserved,0.000000


In [ ]:
# df = pd.DataFrame(rows)

# df.to_csv(r"D:\VSCode\MapBiomas-Award\MapBiomas_Area_ByBiome_ByTrend_Rx5day_1987-2023_proj.csv", index=False)

# df

# Plot figure

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import os
from shapely.geometry import box

# Configurações de Path e Estilo
BASE_PATH = r"D:\VSCode\MapBiomas-Award"
INDICES = ["Rx5day", "R99pTOT", "R20"]
BIOMES = ["Amazon", "Caatinga", "Cerrado", "Atlantic Forest", "Pampa", "Pantanal"]
YEARS = np.arange(1987, 2024)
GROUPS = ["forest", "farming", "otherVegetation", "nonVegetated", "water", "notObserved"]
COLOR_MAP = {"forest": "#1f8d49", "farming": "#ffefc3", "otherVegetation": "#d6bc74",
             "nonVegetated": "#d4271e", "water": "#4fa3d1", "notObserved": "#bdbdbd"}

# 1. Carregar dados
data_dict = {}
for indice in INDICES:
    path = os.path.join(BASE_PATH, f"MapBiomas_Area_ByBiome_ByTrend_{indice}_1987-2023.csv")
    data_dict[indice] = pd.read_csv(path)

# 2. Configuração para A4 Portrait (Vertical)
# A4 é ~8.27 x 11.69 polegadas.
fig, axes = plt.subplots(len(BIOMES), len(INDICES), figsize=(8.27, 11.69), sharex='col', sharey=False)

for row_idx, biome in enumerate(BIOMES):
    # Cálculo do limite de eixo para a linha
    max_area_row = 0
    for indice in INDICES:
        df_temp = data_dict[indice]
        pico = df_temp[df_temp['biome'] == biome].groupby(['year', 'trend_sign'])['area_km2'].sum().max()
        if pico > max_area_row:
            max_area_row = pico
    
    y_limit = max_area_row * 1.3  # Aumentei a margem para 30% para caber os boxes de texto

    for col_idx, indice in enumerate(INDICES):
        ax = axes[row_idx, col_idx]
        df_b = data_dict[indice][data_dict[indice]["biome"] == biome]
        
        for trend in ["positive", "negative"]:
            df_t = df_b[df_b["trend_sign"] == trend]
            bottom = np.zeros(len(YEARS))

            for g in GROUPS:
                df_g = df_t[df_t['name'] == g].set_index('year').reindex(YEARS, fill_value=0)
                values = df_g["area_km2"].values
                if trend == "negative": values = -values

                ax.bar(np.arange(len(YEARS)), values, 0.9, bottom=bottom, color=COLOR_MAP[g],
                       edgecolor="black", linewidth=0.05, zorder=3)
                bottom += values

        ax.set_ylim(-y_limit, y_limit)
        ax.axhline(0, color="black", linewidth=0.8, zorder=4)
        ax.grid(True, which='major', linestyle='--', alpha=0.3, zorder=0)
        
        # Anotações - Reduzi levemente o fontsize para caber no layout A4 vertical
        props_pos = dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='#1f78b4', linewidth=0.5)
        props_neg = dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='#d93025', linewidth=0.5)

        # Notas menores (fontsize 8-9) funcionam melhor em subplots densos
        ax.text(0.03, 0.96, "Positive Rainfall Trend Areas", transform=ax.transAxes, fontsize=8, 
                color="#1f78b4", fontweight='bold', va='top', bbox=props_pos, zorder=5)
        ax.text(0.03, 0.04, "Negative Rainfall Trend Areas", transform=ax.transAxes, fontsize=8, 
                color="#d93025", fontweight='bold', va='bottom', bbox=props_neg, zorder=5)

        if row_idx == 0:
            ax.set_title(indice, fontsize=12, weight="bold", pad=10)
        if col_idx == 0:
            ax.set_ylabel(f"{biome}\n(km²)", fontsize=10, weight="bold")

        ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{abs(y)/1e3:.0f}k" if abs(y) >= 1e3 else f"{abs(y):.0f}"))
        ax.tick_params(axis='y', which='major', labelsize=8)

# Eixo X apenas na última linha
for ax in axes[-1, :]:
    ax.set_xticks(np.arange(0, len(YEARS), 9)) # Intervalos de 10 anos para clareza
    ax.set_xticklabels(YEARS[::9], fontsize=9)

# Legenda centralizada na parte inferior
labels = ['Forest', 'Farming', 'Other Vegetation.', 'Non Vegetated', 'Water', 'Not Observed']
handles = [plt.Rectangle((0, 0), 1, 1, color=COLOR_MAP[g]) for g in GROUPS]
fig.legend(
    handles, labels,
    loc="lower center",
    ncol=6,
    frameon=False,
    fontsize=10,
    bbox_to_anchor=(0.5, 0.01)
)

# Ajuste fino dos espaços
# rect=[esquerda, baixo, direita, cima]
plt.tight_layout(rect=[0, 0.03, 1, 1], pad=0.8)

plt.savefig(r'D:\VSCode\MapBiomas-Award\Figs\barPlots_A4_Vertical.png', dpi=300, bbox_inches='tight')
plt.show()